# Trích xuất Khung xương (BlazePose) - Kaggle

- Dùng Kaggle Dataset làm Input, xuất ra /kaggle/working và nén zip.

In [ ]:
# Cell 1: Clone mã nguồn
import os
REPO   = "https://github.com/tuan8p/Skeleton-EAA-Pose.git"
BRANCH = "dai"
WORKDIR = "/kaggle/working/Skeleton-EAA-Pose"
if not os.path.isdir(WORKDIR):
    !git clone -b {BRANCH} {REPO} {WORKDIR}
else:
    !git -C {WORKDIR} pull origin {BRANCH}
%cd {WORKDIR}
# %cd d:\Downloads\ĐATN\Skeleton-EAA-Pose

d:\Downloads\ĐATN\Skeleton-EAA-Pose


In [ ]:
# Cell 2: Cài đặt thư viện
!pip install -q mediapipe opencv-python numpy scipy pyyaml tqdm psutil
# !pip install -r requirements.txt
# !pip install ipywidgets



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Cell 3: Cấu hình hệ thống & Tự động tải Model
import os
from src.config_manager import ConfigManager

DATASET = "PKU"   # Chọn "PKU" hoặc "TSU"

DETECTION_OUTPUT_DIR = "/kaggle/input/datasets/tuan8p/bboxes-detection"
# DETECTION_OUTPUT_DIR = r"D:\Downloads\ĐATN\outputs_detection"
POSE_OUTPUT_DIR = "/kaggle/working/outputs_pose"

PKU_PATHS = {
    "video_dir":      "/kaggle/input/datasets/tuan8p/pku-rgb-p2",
    "annotation_dir": "/kaggle/input/datasets/tuan8p/pku-annotation",
}

TSU_PATHS = {
    "video_dir":      "/kaggle/input/datasets/tuan8p/tsu-rgb/Videos_mp4",
    "annotation_dir": "/kaggle/input/datasets/tuan8p/tsu-annotation/Annotation_v1.0/Annotation",
    # "video_dir":      r"D:\Downloads\ĐATN\TSU-RGB",
    # "annotation_dir": r"D:\Downloads\ĐATN\Annotation_v1.0",
}

# --- Tự động chọn thư mục tương ứng --- 
if DATASET == "PKU":
    VIDEO_INPUT_DIR = PKU_PATHS["video_dir"]
    ANN_INPUT_DIR   = PKU_PATHS["annotation_dir"]
else:
    VIDEO_INPUT_DIR = TSU_PATHS["video_dir"]
    ANN_INPUT_DIR   = TSU_PATHS["annotation_dir"]
os.makedirs(POSE_OUTPUT_DIR, exist_ok=True)

# ==========================================
# CẤU HÌNH PIPELINE (Chỉnh sửa ở đây)
# ==========================================
SETTINGS = {
    # Các tuỳ chọn loại mô hình: pose_landmarker_lite.task, _full.task, _heavy.task
    "mediapipe.model_path": "models/pose_landmarker_heavy.task", 
    
    # Các ngưỡng tự tin của BlazePose (Tăng lên nếu muốn bỏ qua nhiễu)
    "mediapipe.min_detection_confidence": 0.5,
    "mediapipe.min_presence_confidence": 0.5,
    "mediapipe.min_tracking_confidence": 0.5,
    
    # Cơ chế Fallback và Nội suy (Temporal)
    "temporal.bbox_interp_max_gap": 10,     # Nội suy tối đa 6 frame nếu mất Bbox
    "temporal.empty_run_frames": 30,       # Nếu mất quá 30 frame (1s), coi là đi ra ngoài
    
    # Đầu ra toạ độ
    "output.coordinate_mode": "world",     # "world" (3D thực tế) hoặc "pixel"
    
    # Hệ thống chạy
    "runtime.num_workers": 4,
}

cfg = ConfigManager("config.yaml")
cfg.set("dataset", DATASET)
cfg.set("paths.video_dir", VIDEO_INPUT_DIR)
cfg.set("paths.annotation_dir", ANN_INPUT_DIR)
cfg.set("detection.output_dir", DETECTION_OUTPUT_DIR)
cfg.set("paths.output_dir", POSE_OUTPUT_DIR)

# Nạp Setting
for k, v in SETTINGS.items():
    cfg.set(k, v)

cfg.save("config.pose_runtime.yaml")
print(f"[OK] Đã lưu cấu hình!")

# --- Tự động tải Model BlazePose theo cấu hình --- 
import urllib.request
model_path = cfg.get("mediapipe.model_path", "models/pose_landmarker_full.task")
model_name = os.path.basename(model_path).replace(".task", "")
url = f"https://storage.googleapis.com/mediapipe-models/pose_landmarker/{model_name}/float16/latest/{model_name}.task"
os.makedirs(os.path.dirname(model_path) or ".", exist_ok=True)
if not os.path.exists(model_path):
    print(f"Đang tải {model_name}.task...")
    urllib.request.urlretrieve(url, model_path)
print(f"[OK] BlazePose model ready: {model_path}")

[OK] Đã lưu cấu hình!
[OK] BlazePose model ready: models/pose_landmarker_heavy.task


In [ ]:
# Cell 4: Chạy trích xuất khung xương (Đa luồng CPU Kaggle)
import os
from src.pipeline_orchestrator import PipelineOrchestrator
import concurrent.futures
from tqdm.notebook import tqdm

orch = PipelineOrchestrator(cfg=cfg)
videos = orch.reader.list_videos()
print(f"Tổng số video cần xử lý: {len(videos)}")

# Chia cắt danh sách video cho các tài khoản:
START, END = 250, 500
videos = videos[START:END]
print(f"Tài khoản này sẽ chạy {len(videos)} video (Từ index {START} đến {END-1}).")

NUM_THREADS = int(cfg.get("runtime.num_workers", 4))
print(f"Bắt đầu bóc xương với {NUM_THREADS} luồng song song...")

def process_single_video(vname):
    if orch.progress.is_video_done(vname):
        return f"Bỏ qua {vname} (Đã xong)"
    try:
        stats = orch.process_video(vname, disable_pbar=True)
        orch.progress.mark_video_done(vname, stats.to_dict())
        failed = stats.total_frames - stats.ok_frames
        return f"Xong {vname} | Thành công: {stats.ok_frames}/{stats.total_frames} | Fail: {failed}"
    except Exception as e:
        return f"Lỗi {vname}: {str(e)}"

# with concurrent.futures.ThreadPoolExecutor(max_workers=NUM_THREADS) as executor:
#     futures = {executor.submit(process_single_video, v): v for v in videos}
#     for future in tqdm(concurrent.futures.as_completed(futures), total=len(videos)):
#         print(future.result())
# print("[HOÀN TẤT] Quá trình bóc xương kết thúc!")

Tổng số video cần xử lý: 536
Tài khoản này sẽ chạy 250 video (Từ index 250 đến 499).
Bắt đầu bóc xương với 10 luồng song song...


  0%|          | 0/250 [00:00<?, ?it/s]

Bỏ qua P12T12C07 (Đã xong)
Bỏ qua P12T11C07 (Đã xong)
Bỏ qua P12T13C07 (Đã xong)
Bỏ qua P12T12C06 (Đã xong)
Bỏ qua P12T11C06 (Đã xong)
Xong P12T13C06 | Thành công: 3864/3864 | Fail: 0


In [ ]:
# # Cell 5: Nén kết quả thành file zip để tải về
# import shutil
# import os

# zip_name = "/kaggle/working/pose_results_tsu"
# print(f"Đang nén thư mục {POSE_OUTPUT_DIR}...")
# shutil.make_archive(zip_name, 'zip', POSE_OUTPUT_DIR)
# print(f"[OK] Đã tạo file: {zip_name}.zip ({os.path.getsize(zip_name+'.zip') // (1024*1024)} MB)")
# print("-> BẠN CÓ THỂ TẢI TỪ CỘT BÊN PHẢI (Mục Output).")

In [ ]:
# =========================================================
# CHẠY LẠI CÁC VIDEO CỤ THỂ (Dùng để vét các file bị rỗng/thiếu)
# =========================================================
import os
import concurrent.futures
from tqdm.notebook import tqdm
from src.pipeline_orchestrator import PipelineOrchestrator
from src.config_manager import ConfigManager

# 1. Danh sách các video cụ thể cần chạy lại
target_videos = [
    # "0003-R", "0005-L", "0007-L", "0009-L", "0009-M",
    # "0009-R", "0011-L", "0012-M", "0014-R", "0015-L",
    # "0016-M", "0017-M", "0017-R", "0018-M", "0018-R",
    # "0019-L", "0023-L", "0023-M", "0027-L", "0036-L",
    # "0038-R", "0039-M", "0041-L", "0041-M", "0041-R",
    # "0043-L", "0044-L", "0044-R", "0051-R", "0052-L",
    # "0053-M", "0057-M", "0058-L", "0059-M", "0069-L",
    # "0069-M", "0072-L", "0073-M", "0077-L", "0078-M",
    # "0079-L", "0079-R", "0080-M", "0080-R", "0081-L",
    # "0089-R", "0090-M", "0090-R", "0103-L", "0103-M",
    # "0107-M", "0107-R", "0108-R", "0109-L", "0114-M",
    # "0117-R", "0118-M", "0121-R", "0122-L", "0122-M",
    # "0123-L", "0141-L", "0144-L", "0149-L", "0149-M",
    # "0153-R", "0173-L", "0177-M", "0179-R", 
    
    "0030-R", "0031-R", "0032-L", "0061-M", "0061-R", 
    "0063-M", "0092-M", "0094-L", "0125-M", "0159-R",

    # "0184-M", "0189-L", "0190-L", "0193-M",
    # "0217-L", "0218-M", "0218-R", "0219-L", "0219-M",
    # "0220-L", "0220-R", "0221-L", "0221-M", "0222-L",
    # "0222-R", "0223-R", "0225-L", "0227-M", "0228-R",
    # "0229-R", "0232-R", "0233-M", "0233-R", "0237-L",
    # "0238-M", "0238-R", "0239-L", "0239-M", "0241-M",
    # "0241-R", "0242-M", "0256-L", "0257-L", "0257-M",
    # "0257-R", "0258-L", "0259-R", "0260-M", "0268-M",
    # "0269-L", "0275-R", "0276-M", "0277-L", "0287-R", 
    # "0288-R", "0290-L", "0291-L", "0291-M", "0291-R", 

    "0210-M", "0243-M", "0244-L", "0279-L", "0279-R", 
    "0292-L", "0292-M", "0294-M", 
    "0295-L", "0295-R", "0297-L", "0297-M", "0301-M", 
    "0303-L", "0303-M", "0305-L", "0307-M", "0307-R", 
    "0308-M", "0311-L", "0315-L", "0316-M", "0316-R", 
    "0317-M", "0318-L", "0348-L", "0354-M", "0356-M", 
    "0357-L", "0357-M", "0358-L", "0360-L", "0361-L",

    # "P12T11C06", "P12T11C07", 
    # "P12T12C06", "P12T12C07", 
    # "P12T13C06", "P12T13C07"
]

# 2. Đọc lại cấu hình đã được khai báo ở các Cell trước
cfg = ConfigManager("config.pose_runtime.yaml")

# 3. Tạo bộ điều phối (PipelineOrchestrator bản mới đã fix lỗi đa luồng)
orch = PipelineOrchestrator(cfg=cfg)

def process_single_video(v):
    # orch.process_video sẽ tự động gọi mô hình riêng cho từng luồng
    orch.process_video(v)

num_workers = cfg.get('runtime.num_workers', 1)
print(f"Tổng số video cần chạy vét: {len(target_videos)}")
print(f"Bắt đầu bóc xương lại với {num_workers} luồng song song...")

# 4. Kích hoạt bóc xương
if num_workers > 1:
    with concurrent.futures.ThreadPoolExecutor(max_workers=num_workers) as executor:
        list(tqdm(executor.map(process_single_video, target_videos), total=len(target_videos)))
else:
    for v in tqdm(target_videos):
        process_single_video(v)
